## Librerías

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cargar y ejecutar un modelo LLM pequeño

## Selección del modelo adecuado
En esta práctica utilizaremos el `modelo Qwen2.5-1.5B-Instruct`, un modelo de lenguaje optimizado para seguir instrucciones. Este modelo ofrece un buen equilibrio entre rendimiento y consumo de memoria, permitiendo su uso en CPU.

##  Cargar el modelo
En este apartado cargaremos el modelo y su tokenizer desde Hugging Face. También seleccionaremos automáticamente la precisión más adecuada según el hardware disponible, usando bfloat16 cuando sea posible para reducir el consumo de memoria.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the best dtype for the current hardware
# bfloat16 is preferred (half memory, stable on CPU)
# float32 is the safe fallback
if torch.cuda.is_available():
    dtype = torch.bfloat16
    device_map = "auto"
elif hasattr(torch, "bfloat16"):
    dtype = torch.bfloat16
    device_map = "cpu"
else:
    dtype = torch.float32
    device_map = "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype = dtype,
    device_map = device_map,
)

print(f"Model loaded on {model.device} with dtype {dtype}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

## Verificación del funcionamiento del modelo
En este apartado comprobamos que el modelo funciona correctamente generando una salida simple. También verificamos que no haya errores numéricos (NaN) en los valores internos del modelo.

In [ ]:
# Quick sanity check
test_input = tokenizer("The capital of France is", return_tensors = "pt").to(model.device)

with torch.no_grad():
    logits = model(**test_input).logits
    last_logits = logits[0, -1, :]
    print(f"Logits — min: {last_logits.min().item():.2f}, max: {last_logits.max().item():.2f}")
    print(f"Contains NaN: {torch.isnan(last_logits).any().item()}")
    
    # Generate a few tokens
    out = model.generate(**test_input, max_new_tokens = 10, do_sample = False,
                         pad_token_id = tokenizer.eos_token_id)
    print(f"Output: {tokenizer.decode(out[0], skip_special_tokens = True)}")

## Comprensión de la tokenización
Antes de procesar texto, el modelo lo convierte en tokens, que son representaciones numéricas. El tokenizer se encarga de transformar el texto en estos tokens y de aplicar el formato necesario para el modelo.

## Basic tokenization
text = "Hello, how are you doing today?"
tokens = tokenizer(text, return_tensors = "pt")

print(f"Input text: {text}")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Number of tokens: {tokens['input_ids'].shape[1]}")
print(f"Decoded tokens: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")

## Generación de texto
En este apartado utilizamos el método `generate()` para producir texto a partir de una entrada. También introducimos parámetros que controlan el comportamiento del modelo, como la aleatoriedad y la diversidad de las respuestas.

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens = 256):
    """
    Generate a response from the model given a raw text prompt.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        prompt: Input text string
        max_new_tokens: Maximum number of tokens to generate
    
    Returns:
        str: The generated response
    """
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            temperature = 0.7,
            top_p = 0.9,
            top_k = 50,
            pad_token_id = tokenizer.eos_token_id,
        )
    
    # Decode only the new tokens (exclude the prompt)
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens = True
    )
    return response


# Test
response = generate_response(model, tokenizer, "The capital of France is")
print(response)

## Uso de plantillas de chat
Los modelos ajustados para instrucciones utilizan un formato de conversación con roles (system, user, assistant). El tokenizer aplica automáticamente este formato mediante plantillas de chat para que el modelo entienda mejor la entrada.

In [ ]:
def chat(model, tokenizer, user_message, system_message = "You are a helpful assistant.",
         max_new_tokens = 256, temperature = 0.7, top_p = 0.9, top_k = 50):
    """
    Send a message using the model's chat template.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        user_message: The user's message
        system_message: The system prompt
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (use 0.0 for greedy)
        top_p: Nucleus sampling threshold
        top_k: Top-k sampling (limits vocabulary at each step)
    
    Returns:
        str: The assistant's response
    """
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, tokenize = False, add_generation_prompt = True
    )
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    
    gen_kwargs = dict(
        max_new_tokens = max_new_tokens,
        pad_token_id = tokenizer.eos_token_id,
    )
    
    if temperature < 0.01:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p
        gen_kwargs["top_k"] = top_k
    
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens = True
    )
    return response


# Test
response = chat(model, tokenizer, "Explain what a neural network is in 2 sentences.")
print(response)

## Parámetros de generación
El comportamiento del modelo al generar texto puede controlarse mediante parámetros como la temperatura, top-p y top-k. Estos permiten ajustar el nivel de aleatoriedad y diversidad en las respuestas generadas.

In [ ]:
# Compare different temperature values
prompt = "Write a creative name for a coffee shop."

print("=== Temperature 0.3 (near deterministic) ===")
for i in range(3):
    r = chat(model, tokenizer, prompt, max_new_tokens = 30, temperature = 0.3)
    print(f"  Run {i + 1}: {r.strip()}")

print("\n=== Temperature 0.7 (balanced) ===")
for i in range(3):
    r = chat(model, tokenizer, prompt, max_new_tokens = 30, temperature = 0.7)
    print(f"  Run {i + 1}: {r.strip()}")

print("\n=== Temperature 1.2 (creative) ===")
for i in range(3):
    r = chat(model, tokenizer, prompt, max_new_tokens = 30, temperature = 1.2)
    print(f"  Run {i + 1}: {r.strip()}")

# Fundamentos del Prompt Engineering
En esta parte veremos cómo diseñar y mejorar las entradas que damos a un modelo de lenguaje para obtener respuestas más precisas y útiles.

La idea principal es que podemos mejorar mucho el resultado sin modificar el modelo, simplemente cambiando la forma en la que le damos la información y la tarea.

## The Anatomy of a Good Prompt

## Zero-Shot Prompting

In [ ]:
# Zero-shot sentiment analysis
zero_shot_prompt = """Classify the sentiment of the following review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The food was absolutely delicious and the service was impeccable. 
Best restaurant experience I've had in years!"

Sentiment:"""

response = chat(model, tokenizer, zero_shot_prompt, max_new_tokens = 10, temperature = 0.0)
print(f"Sentiment: {response.strip()}")

## Chain-of-Thought (CoT) Prompting

In [ ]:
# Without Chain-of-Thought
direct_prompt = """If a store sells apples for 2 euros each and oranges for 3 euros each,
and Maria buys 4 apples and 5 oranges, how much does she pay in total?

Answer:"""

response_direct = chat(model, tokenizer, direct_prompt, max_new_tokens = 50, temperature = 0.0)
print(f"Direct answer: {response_direct.strip()}")

# With Chain-of-Thought
cot_prompt = """If a store sells apples for 2 euros each and oranges for 3 euros each,
and Maria buys 4 apples and 5 oranges, how much does she pay in total?

Let's solve this step by step:"""

response_cot = chat(model, tokenizer, cot_prompt, max_new_tokens = 150, temperature = 0.0)
print(f"\nChain-of-Thought answer:\n{response_cot.strip()}")

## Structured Output Prompting

## 3.7. Role Prompting

In [ ]:
# Same question, different roles
question = "Explain what an API is."

# Role 1: Expert for developers
response_expert = chat(
    model, tokenizer, question,
    system_message = "You are a senior software architect. Give precise, technical explanations.",
    max_new_tokens = 150, temperature = 0.3
)
print("=== Expert Explanation ===")
print(response_expert.strip())

# Role 2: Teacher for beginners
response_teacher = chat(
    model, tokenizer, question,
    system_message = "You are a patient teacher explaining concepts to a 10-year-old. Use simple words and analogies.",
    max_new_tokens = 150, temperature = 0.3
)
print("\n=== Beginner Explanation ===")
print(response_teacher.strip())

# 4. Part 3: Prompt Chaining

## 4.1. Breaking Complex Tasks into Steps

In [ ]:
def prompt_chain_analysis(model, tokenizer, text):
    """
    Analyze a text through a chain of prompts:
    1. Summarize the text
    2. Extract key entities
    3. Determine the overall sentiment
    4. Generate a final report
    """
    # Step 1: Summarize
    summary = chat(
        model, tokenizer,
        f"Summarize the following text in 2 sentences:\n\n{text}",
        max_new_tokens = 100, temperature = 0.3
    )
    print(f"Step 1 — Summary:\n{summary.strip()}\n")
    
    # Step 2: Extract entities
    entities = chat(
        model, tokenizer,
        f"From the following summary, list the key entities (people, organizations, locations) as a comma-separated list:\n\n{summary}",
        max_new_tokens = 100, temperature = 0.0
    )
    print(f"Step 2 — Entities:\n{entities.strip()}\n")
    
    # Step 3: Sentiment
    sentiment = chat(
        model, tokenizer,
        f"What is the overall sentiment of this text? Answer with one word (POSITIVE, NEGATIVE, or NEUTRAL):\n\n{summary}",
        max_new_tokens = 10, temperature = 0.0
    )
    print(f"Step 3 — Sentiment:\n{sentiment.strip()}\n")
    
    # Step 4: Final report
    report = chat(
        model, tokenizer,
        f"""Generate a brief analytical report given the following information:
Summary: {summary}
Key Entities: {entities}
Sentiment: {sentiment}

Write a 3-sentence report:""",
        max_new_tokens = 150, temperature = 0.3
    )
    print(f"Step 4 — Final Report:\n{report.strip()}")
    
    return {
        "summary": summary.strip(),
        "entities": entities.strip(),
        "sentiment": sentiment.strip(),
        "report": report.strip()
    }


# Test with a sample text
sample_text = """
The European Commission announced today a landmark agreement on artificial intelligence regulation.
The AI Act, which has been under negotiation for over three years, establishes a risk-based framework
for AI systems deployed across the European Union. Commissioner Thierry Breton stated that this
regulation will serve as a global benchmark. Technology companies including Google, Microsoft, and
Meta have expressed mixed reactions, with some praising the clarity it brings while others worry
about potential impacts on innovation. The regulation is expected to take full effect by 2026.
"""

result = prompt_chain_analysis(model, tokenizer, sample_text)

# 5. Part 4: Evaluating Prompt Strategies

## 5.1. Building an Evaluation Framework

In [ ]:
# Define a small evaluation dataset
eval_dataset = [
    {
        "question": "What is the capital of Japan?",
        "expected": "Tokyo",
        "category": "factual"
    },
    {
        "question": "If a shirt costs 25 euros and is on sale for 20% off, what is the sale price?",
        "expected": "20",
        "category": "math"
    },
    {
        "question": "Sort these numbers from smallest to largest: 7, 2, 9, 1, 5",
        "expected": "1, 2, 5, 7, 9",
        "category": "reasoning"
    },
    {
        "question": "What is the sentiment of this sentence: 'I absolutely hated every minute of that terrible movie.'",
        "expected": "NEGATIVE",
        "category": "classification"
    },
    {
        "question": "A farmer has 17 sheep. All but 9 die. How many sheep are left?",
        "expected": "9",
        "category": "reasoning"
    },
    {
        "question": "Translate to French: 'The weather is beautiful today.'",
        "expected": "Il fait beau aujourd'hui",
        "category": "translation"
    },
]

def evaluate_strategy(model, tokenizer, dataset, prompt_fn, strategy_name):
    """
    Evaluate a prompting strategy on the dataset.
    """
    results = []
    
    for item in dataset:
        prompt = prompt_fn(item["question"])
        response = chat(model, tokenizer, prompt, max_new_tokens = 50, temperature = 0.0)
        response_clean = response.strip().lower()
        expected_clean = item["expected"].lower()
        
        correct = expected_clean in response_clean
        
        results.append({
            "question": item["question"],
            "expected": item["expected"],
            "response": response.strip(),
            "correct": correct,
            "category": item["category"]
        })
    
    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\n{'=' * 60}")
    print(f"Strategy: {strategy_name}")
    print(f"Overall Accuracy: {accuracy:.1%}")
    print(f"{'=' * 60}")
    
    for r in results:
        status = "✓" if r["correct"] else "✗"
        print(f"  [{status}] {r['category']:15s} | Expected: {r['expected']:20s} | Got: {r['response'][:40]}")
    
    return {"strategy": strategy_name, "accuracy": accuracy, "details": results}

## 5.2. Comparing Strategies

In [ ]:
# Strategy 1: Direct (zero-shot, no special instructions)
def direct_prompt(question):
    return f"{question}\n\nAnswer concisely:"

# Strategy 2: Instructed (with clear formatting instructions)
def instructed_prompt(question):
    return f"""Answer the following question. Be concise and give ONLY the answer, no explanation.

Question: {question}

Answer:"""

# Strategy 3: Chain-of-Thought
def cot_prompt(question):
    return f"""Answer the following question. Think step by step, then provide your final answer 
on the last line prefixed with "ANSWER:".

Question: {question}

Step-by-step reasoning:"""


# Run evaluation
results_direct = evaluate_strategy(model, tokenizer, eval_dataset, direct_prompt, "Direct")
results_instructed = evaluate_strategy(model, tokenizer, eval_dataset, instructed_prompt, "Instructed")
results_cot = evaluate_strategy(model, tokenizer, eval_dataset, cot_prompt, "Chain-of-Thought")

## 5.3. Visualizing Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

strategies = [results_direct, results_instructed, results_cot]
strategy_names = [r["strategy"] for r in strategies]
accuracies = [r["accuracy"] for r in strategies]

fig, axes = plt.subplots(1, 2, figsize = (14, 5))

# Plot 1: Overall accuracy
bars = axes[0].bar(strategy_names, accuracies, color = ['#e41a1c', '#377eb8', '#4daf4a'])
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Overall Accuracy by Prompting Strategy')
axes[0].set_ylim(0, 1.0)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.02,
                 f'{acc:.0%}', ha = 'center', fontweight = 'bold')

# Plot 2: Per-category accuracy
categories = list(set(item["category"] for item in eval_dataset))
x = np.arange(len(categories))
width = 0.25

for i, result in enumerate(strategies):
    cat_acc = {}
    for detail in result["details"]:
        cat = detail["category"]
        if cat not in cat_acc:
            cat_acc[cat] = []
        cat_acc[cat].append(detail["correct"])
    
    cat_accuracies = [np.mean(cat_acc.get(cat, [0])) for cat in categories]
    axes[1].bar(x + i * width, cat_accuracies, width, label = result["strategy"])

axes[1].set_xlabel('Category')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy by Category and Strategy')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(categories, rotation = 45, ha = 'right')
axes[1].legend()
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig("prompt_strategy_comparison.png", dpi = 150)
plt.show()